# Conversation Data Inspector
Quick random sample of N conversations across datasets.

In [1]:
import os, random
import pyarrow.parquet as pq

ROOT = os.path.abspath('../data')

DATASETS = {
    'smoltalk':      f'{ROOT}/conversation_data',
    'ultrachat':     f'{ROOT}/conversation_data',
    'lima':          f'{ROOT}/lima_data',
    'mmlu':          f'{ROOT}/mmlu_data',
    'gsm8k':         f'{ROOT}/gsm8k_data',
    'topic_struct':  f'{ROOT}/custom_data',
    'topic_switch':  f'{ROOT}/topic_switch_data',
}

SOURCE_FILTER = {
    'smoltalk':     lambda s: s == 'smoltalk',
    'ultrachat':    lambda s: s.startswith('ultrachat'),
    'lima':         lambda s: s == 'lima',
    'mmlu':         lambda s: s == 'mmlu',
    'gsm8k':        lambda s: s == 'gsm8k',
    'topic_struct': lambda s: s == 'topic_structure',
    'topic_switch': lambda s: s == 'topic_switch',
}

def load_rows(name):
    path = DATASETS[name]
    rows = []
    for f in sorted(os.listdir(path)):
        if not f.endswith('.parquet'): continue
        for row in pq.read_table(f'{path}/{f}').to_pylist():
            if SOURCE_FILTER[name](row.get('source', '')):
                rows.append(row)
    return rows

print('Datasets loaded. Set N and dataset below.')

Datasets loaded. Set N and dataset below.


In [5]:
# ── Config ─────────────────────────────────────────────────────────────────
DATASET = 'topic_switch'   # smoltalk | ultrachat | lima | mmlu | gsm8k | topic_struct | topic_switch
N       = 5            # number of conversations to sample
PREVIEW = 12           # words to show per message
# ───────────────────────────────────────────────────────────────────────────

def preview(text, n=PREVIEW):
    words = text.split()
    snippet = ' '.join(words[:n])
    return snippet + ('...' if len(words) > n else '')

def show(row, idx):
    src = row.get('source', '')
    subj = f"  [{row['subject']}]" if 'subject' in row else ''
    print(f'── #{idx}  source={src}{subj}')
    for msg in row['messages']:
        role = msg['role'].upper()[:6].ljust(6)
        print(f'  {role}  {preview(msg["content"])}')
    print()

rows = load_rows(DATASET)
sample = random.sample(rows, min(N, len(rows)))
print(f'{DATASET}: {len(rows)} total rows, showing {len(sample)}\n')
for i, row in enumerate(sample, 1):
    show(row, i)

topic_switch: 99788 total rows, showing 5

── #1  source=topic_switch
  USER    A family of 12 monkeys collected 10 piles of bananas. 6 piles...
  ASSIST  The first 6 bunches had 6 x 9 x 14 = <<6*9*14=756>>756...
  USER    Gerald had 20 toy cars. He donated 1/4 of his toy cars...
  ASSIST  Gerald gave away 20 x 1/4 = <<20*1/4=5>>5 of his toy cars....
  USER    Bert fills out the daily crossword puzzle in the newspaper every day....
  ASSIST  At 7 days a week, it takes Bert 2 * 7 =...
  USER    Jacob takes four tests in his physics class and earns 85, 79,...
  ASSIST  Since wants needs an 85 as an average and he takes five...
  USER    Johnson has a sack of potatoes. He gives Gina 69 potatoes, gives...
  ASSIST  He gives twice 69 potatoes to Tom, which is 69*2 = <<69*2=138>>138...
  USER    Roman and Remy took separate showers. Remy used 1 more gallon than...
  ASSIST  Let R = Roman's gallons Remy = 3R + 1 4R +...
  USER    The P.T.O. decided to provide shirts for the elementary student